# Data Wrangling

This project focuses on applying data wrangling techniques to the Netflix Movies and TV Shows dataset obtained from Kaggle.

The objective is to Load, explore, clean, transform, validate, and export the dataset using Python and the pandas library.

In [206]:
# Importing the libraries
import pandas as pd
import datetime as dt

### Loading the Dataset

In [207]:
filepath = 'C:\\Users\\JUSTUS ONYANGO\\OneDrive\\Data Wrangling\\netflix_titles.csv'
df = pd.read_csv(filepath)
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


### Discovery

In [208]:
#Have a quick overview of the data
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB


In [209]:
# Number of rows and columns
print("Shape of the dataset (R x C):", df.shape)

Shape of the dataset (R x C): (8807, 12)


In [210]:
# List of all column names
print("Columns in the dataset:\n", df.columns.tolist())

Columns in the dataset:
 ['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in', 'description']


In [211]:
# Data types of each column
print("Data types:\n", df.dtypes)

Data types:
 show_id         object
type            object
title           object
director        object
cast            object
country         object
date_added      object
release_year     int64
rating          object
duration        object
listed_in       object
description     object
dtype: object


In [212]:
# Group and Count of missing (null) values in each column
print("Missing values per column:\n", df.isnull().sum())

Missing values per column:
 show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64


In [213]:
# Group and Count of duplicate rows
print("Number of duplicate rows:", df.duplicated().sum())

Number of duplicate rows: 0


### Structuring

In [214]:
# Convert 'date_added' to datetime
df['date_added'] = pd.to_datetime(df['date_added'],format='mixed')
df.dtypes

show_id                 object
type                    object
title                   object
director                object
cast                    object
country                 object
date_added      datetime64[ns]
release_year             int64
rating                  object
duration                object
listed_in               object
description             object
dtype: object

In [215]:
# Separate 'duration' into numeric value and unit (e.g., '90 min' → 90, 'min')
df[['duration_value', 'duration_unit']] = df['duration'].str.extract(r'(\d+)\s*(\w+)')

In [216]:
# Convert duration_value to numeric
df['duration_value'] = pd.to_numeric(df['duration_value'])

In [217]:
# View Resulting columns
print(df[['duration_value', 'duration_unit']])

      duration_value duration_unit
0               90.0           min
1                2.0       Seasons
2                1.0        Season
3                1.0        Season
4                2.0       Seasons
...              ...           ...
8802           158.0           min
8803             2.0       Seasons
8804            88.0           min
8805            88.0           min
8806           111.0           min

[8807 rows x 2 columns]


In [218]:
# Split 'cast' column into individual actors and create a new DataFrame
cast_df = df[['show_id', 'cast']].copy()
cast_df['cast'] = cast_df['cast'].str.split(', ')
cast_df = cast_df.explode('cast')
cast_df = cast_df.rename(columns={'cast': 'actor'})
print(cast_df.head())


  show_id           actor
0      s1             NaN
1      s2      Ama Qamata
1      s2     Khosi Ngema
1      s2   Gail Mabalane
1      s2  Thabang Molaba


In [219]:
genres_df = df[['show_id', 'listed_in']].copy()
genres_df['listed_in'] = genres_df['listed_in'].str.split(', ')
genres_df = genres_df.explode('listed_in')
genres_df = genres_df.rename(columns={'listed_in': 'genre'})
print(genres_df.head())

  show_id                   genre
0      s1           Documentaries
1      s2  International TV Shows
1      s2               TV Dramas
1      s2            TV Mysteries
2      s3          Crime TV Shows


### Cleaning

In [220]:
# Check for duplicate rows
print("Duplicate rows before:", df.duplicated().sum())

Duplicate rows before: 0


In [221]:
# Drop duplicate rows if any
#df = df.drop_duplicates()

In [222]:
# Drop description column because it will not be used
df = df.drop(columns=['description'])

In [223]:
# Before imputation
missing_before = df['director'].isna().sum()
# Step 1: Identify frequent Director–Cast pairs
# Combine director and cast for counting
df['dir_cast'] = df['director'].fillna('') + '---' + df['cast'].fillna('')

# Count how often each combination appears
counts = df['dir_cast'].value_counts()

# Keep only combinations that appear at least 3 times
frequent_pairs = counts[counts >= 3].index

# Step 2: Build a mapping: cast string → director
# This assumes a cast string maps to one director (most frequent combos)
cast_to_director = {}
for pair in frequent_pairs:
    director, cast = pair.split('---')
    if cast:  # skip empty casts
        cast_to_director[cast] = director

# Step 3: Impute missing directors based on cast
for cast, director in cast_to_director.items():
    df.loc[(df['director'].isna()) & (df['cast'] == cast), 'director'] = director

# After imputation
missing_after = df['director'].isna().sum()

print(f"Directors missing before: {missing_before}")
print(f"Directors missing after: {missing_after}")
print(f"Directors imputed: {missing_before - missing_after}")


Directors missing before: 2634
Directors missing after: 2589
Directors imputed: 45


In [224]:

# Step 4: Fill remaining missing directors
df['director'] = df['director'].fillna('Not Given')


In [225]:
#Use directors to fill missing countries

# Before imputation
missing_before = df['country'].isna().sum()

# Step 1: Create a mapping: director → most frequent country
director_country_map = df.groupby('director')['country'].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None).to_dict()

# Step 2: Fill missing countries based on director
df['country'] = df.apply(
    lambda row: director_country_map[row['director']] if pd.isna(row['country']) else row['country'], axis=1
)

# Step 3: Optional: Fill remaining missing countries
df['country'] = df['country'].fillna('Not Given')

# After imputation
missing_after = df['country'].isna().sum()

print(f"Countries missing before: {missing_before}")
print(f"Countries missing after: {missing_after}")
print(f"Countries imputed: {missing_before - missing_after}")


Countries missing before: 831
Countries missing after: 0
Countries imputed: 831


In [226]:
# Assign Not Given to all other fields
df.loc[df['cast'].isna(),'cast'] = 'Not Given'

In [227]:
# dropping other row records that are null
df.drop(df[df['date_added'].isna()].index,axis=0,inplace=True)
df.drop(df[df['rating'].isna()].index,axis=0,inplace=True)
df.drop(df[df['duration'].isna()].index,axis=0,inplace=True)

In [228]:
# Group and Count of missing (null) values in each column
print("Missing values per column:\n", df.isnull().sum())

Missing values per column:
 show_id           0
type              0
title             0
director          0
cast              0
country           0
date_added        0
release_year      0
rating            0
duration          0
listed_in         0
duration_value    0
duration_unit     0
dir_cast          0
dtype: int64


#### Errors

In [229]:
# check if there are any added_dates that come before release_year

sum(df['date_added'].dt.year < df['release_year'])
df.loc[(df['date_added'].dt.year < df['release_year']),['date_added','release_year']]

,date_added,release_year
1551,2020-12-14,2021
1696,2020-11-15,2021
2920,2020-02-13,2021
3168,2019-12-06,2020
3287,2019-11-13,2020
3369,2019-10-25,2020
3433,2019-10-11,2020
4844,2018-05-30,2019
4845,2018-05-29,2019
5394,2017-07-01,2018


In [230]:
# Fix inconsistencies: if release_year > date_added year, replace release_year
mask = df['date_added'].dt.year < df['release_year']

df.loc[mask, 'release_year'] = df.loc[mask, 'date_added'].dt.year


In [231]:
# sample some of the records and check that they have been accurately replaced
df.iloc[[1551,1696,2920,3168]]

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,duration_value,duration_unit,dir_cast
1551,s1552,TV Show,Hilda,Not Given,"Bella Ramsey, Ameerah Falzon-Ojo, Oliver Nelso...","United Kingdom, Canada, United States",2020-12-14,2020,TV-Y7,2 Seasons,Kids' TV,2.0,Seasons,"---Bella Ramsey, Ameerah Falzon-Ojo, Oliver Ne..."
1696,s1697,TV Show,Polly Pocket,Not Given,"Emily Tennant, Shannon Chan-Kent, Kazumi Evans...","Canada, United States, Ireland",2020-11-15,2020,TV-Y,2 Seasons,Kids' TV,2.0,Seasons,"---Emily Tennant, Shannon Chan-Kent, Kazumi Ev..."
2920,s2921,TV Show,Love Is Blind,Not Given,"Nick Lachey, Vanessa Lachey",United States,2020-02-13,2020,TV-MA,1 Season,"Reality TV, Romantic TV Shows",1.0,Season,"---Nick Lachey, Vanessa Lachey"
3168,s3169,TV Show,Fuller House,Not Given,"Candace Cameron Bure, Jodie Sweetin, Andrea Ba...",United States,2019-12-06,2019,TV-PG,5 Seasons,TV Comedies,5.0,Seasons,"---Candace Cameron Bure, Jodie Sweetin, Andrea..."


### Validation

In [232]:
# Remove any columns added during wrangling

df.drop(columns=['dir_cast'], inplace=True)

##### Consistency

In [233]:
# Date Logic Check
#Confirm that no more release_year inconsistencies
sum(df['date_added'].dt.year < df['release_year'])

0

In [236]:
# Type vs Duration Unit Check
# Movies should be in minutes, TV Shows in seasons
df.groupby('type')['duration'].value_counts().head(10)


type   duration
Movie  90 min      152
       93 min      146
       94 min      146
       97 min      146
       91 min      144
       95 min      137
       96 min      130
       92 min      129
       102 min     122
       98 min      120
Name: count, dtype: int64

##### Completeness (Missing Data)

In [238]:
df.isnull().sum().sort_values(ascending=False)


show_id           0
type              0
title             0
director          0
cast              0
country           0
date_added        0
release_year      0
rating            0
duration          0
listed_in         0
duration_value    0
duration_unit     0
dtype: int64

### Publish

In [ ]:
# Save as CSV
df.to_csv('C:\\Users\\JUSTUS ONYANGO\\OneDrive\\Data Wrangling\\netflix_titles_cleaned.csv', index=False)